# Final Validation Set Model Evaluation Pipeline

This notebook loads the frozen validation dataset split, queries the MLflow SQLite backend 
to fetch the highest-performing model pipeline artifact, and computes generalization metrics.

In [48]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Environment Configurations & Data Ingestion

In [49]:
# Define core path anchors relative to notebook location
EXPERIMENT_NAME = "customer-churn-optuna"

from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

In [50]:
# Ingest Data Partitions
VAL_FEATURES_PATH = "../data/processed/raw_features/X_val.csv"
VAL_TARGET_PATH = "../data/processed/target/y_val.csv"

In [51]:
# Load validation partitions
X_val = pd.read_csv(VAL_FEATURES_PATH)
y_val = pd.read_csv(VAL_TARGET_PATH).squeeze()

print(f"Validation features shape: {X_val.shape}")
print(f"Validation target distribution:\n{y_val.value_counts(normalize=True)}")

Validation features shape: (1600, 17)
Validation target distribution:
Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64


## 2. Query MLflow Registry for the Absolute Best Run

In [52]:
# Enforce experiment configuration check explicitly against our SQLite DB
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(f"No active model runs found inside experiment: '{EXPERIMENT_NAME}'")

In [53]:
# Search for the best single run within this experiment using target metrics
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.pr_auc DESC", "metrics.test_auc DESC", "metrics.best_auc DESC"],
    max_results=1
)

if runs_df.empty:
    raise RuntimeError(f"No active model runs found inside experiment: '{EXPERIMENT_NAME}'")

In [54]:
best_run = runs_df.iloc[0]

In [55]:
print("=" * 60)
print(f"BEST PIPELINE FOUND IN SQLite DB")
print("=" * 60)
print(f"Model Flavor  : {best_run.get('tags.mlflow.runName', 'Unnamed Model')}")
print(f"MLflow Run ID : {best_run.run_id}")
print(f"Training Score: {best_run.get('metrics.pr_auc', best_run.get('metrics.best_auc', 0.0)):.4f} (PR-AUC)")
print("=" * 60)

BEST PIPELINE FOUND IN SQLite DB
Model Flavor  : 190
MLflow Run ID : a2a6cc84da2942698efc305f8ecb5ee0
Training Score: 0.6977 (PR-AUC)


## 3. Load the Serialized Production Pipeline Model Object

In [56]:
# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)
experiment_id = "1"  # Your experiment ID

# 2. Programmatically query your tracking history to isolate the parent run
runs_df = mlflow.search_runs(experiment_ids=[experiment_id])

# Filter out the children; find the most recent successful parent run
parent_runs = runs_df[runs_df["tags.mlflow.parentRunId"].isna() & (runs_df["status"] == "FINISHED")]
best_parent_run = parent_runs.sort_values(by="metrics.pr_auc", ascending=False).iloc[0]

best_run_id = best_parent_run["run_id"]
best_pr_auc = best_parent_run["metrics.test_pr_auc"]

print(f"Fetching Champion Model from Parent Run ID: {best_run_id}")
print(f"Validated Validation PR-AUC: {best_pr_auc:.4f}")

# 3. Load the model artifact directly back into your environment
model_uri = f"runs:/{best_run_id}/best_model"
best_pipeline = mlflow.sklearn.load_model(model_uri)

print("\nPipeline successfully loaded! Type:", type(best_pipeline))

Fetching Champion Model from Parent Run ID: 3baa83fba10949f99b4d570322b85014
Validated Validation PR-AUC: 0.7341



Pipeline successfully loaded! Type: <class 'sklearn.pipeline.Pipeline'>


## 4. Execute Generalization Inference on Validation Cohort

In [57]:
# Generate hard predictions and raw probability vectors
y_pred = best_pipeline.predict(X_val)
y_proba = best_pipeline.predict_proba(X_val)[:, 1]

## 5. Compute Generalization Performance Metrics

In [58]:
# Calculate precision, recall, and dual area under the curves
precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, y_proba)
pr_auc = average_precision_score(y_val, y_proba)

In [59]:
# Create a clean display frame
metrics_summary = pd.DataFrame({
    "Metric Name": ["Precision", "Recall (Sensitivity)", "ROC-AUC (Ranking)", "PR-AUC (Average Precision)"],
    "Validation Score": [precision, recall, roc_auc, pr_auc]
})

In [60]:
# Display clean tabular layout
print("\n### Final Validation Assessment Profile ###")
display(metrics_summary.style.format({"Validation Score": "{:.4f}"}))


### Final Validation Assessment Profile ###


,Metric Name,Validation Score
0,Precision,0.7009
1,Recall (Sensitivity),0.4601
2,ROC-AUC (Ranking),0.8548
3,PR-AUC (Average Precision),0.6660


### 5.1 Granular Classification Report Matrix

In [61]:
print("\n" + classification_report(y_val, y_pred, zero_division=0))


              precision    recall  f1-score   support

           0       0.87      0.95      0.91      1274
           1       0.70      0.46      0.56       326

    accuracy                           0.85      1600
   macro avg       0.79      0.70      0.73      1600
weighted avg       0.84      0.85      0.84      1600



### 5.2 Confusion Matrix Count Outputs

In [62]:
cm = confusion_matrix(y_val, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Retained (0)', 'Actual Churned (1)'], 
    columns=['Predicted Retained (0)', 'Predicted Churned (1)']
)
display(cm_df)

,Predicted Retained (0),Predicted Churned (1)
Actual Retained (0),1210,64
Actual Churned (1),176,150


train_pr_auc:

			Predicted Retained (0)	Predicted Churned (1)
Actual Retained (0)	1178				96
Actual Churned (1)	156					170


train_pr_auc

            Predicted Retained (0)	Predicted Churned (1)
Actual Retained (0)	1210	            64
Actual Churned (1)	176	                150


In [63]:
from src import mlflow_utils 

mlflow_utils.get_experiment_summary(EXPERIMENT_NAME)

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.pr_auc', 'metrics.train_precision',
       'metrics.test_precision', 'metrics.train_recall',
       'metrics.test_roc_auc', 'metrics.test_recall', 'metrics.test_f1',
       'metrics.train_accuracy', 'metrics.train_f1', 'metrics.train_pr_auc',
       'metrics.train_roc_auc', 'metrics.test_pr_auc', 'metrics.best_auc',
       'metrics.test_accuracy', 'params.xgb_scale_pos_weight',
       'params.xgb_reg_lambda', 'params.model', 'params.xgb_reg_alpha',
       'params.scaler', 'params.xgb_gamma', 'params.xgb_learning_rate',
       'params.xgb_n_estimators', 'params.xgb_colsample_bytree',
       'params.xgb_subsample', 'params.encoder', 'params.rf_max_depth',
       'params.rf_min_samples_leaf', 'params.rf_min_samples_split',
       'params.rf_n_estimators', 'params.n_output_features',
       'params.n_input_features', 'tags.mlflow.user',
       'tags.xgb_reg_alpha_distribution',
       'tag

,run_id,tags.mlflow.runName,start_time
0,871ed2926df6495bbe2efca9388d6002,501,2026-06-18 19:26:19.705000+00:00
1,36d20629cc51439db270efb5f512818d,500,2026-06-18 19:26:19.462000+00:00
2,7ad30bcda33942aba768eb21d9a7d5d1,499,2026-06-18 19:26:19.226000+00:00
3,bd91435f0a28452e8b900d1e3d4cc5e3,498,2026-06-18 19:26:18.975000+00:00
4,f99f35f5a25b453191c83a576dcb24b5,497,2026-06-18 19:26:18.731000+00:00
...,...,...,...
6008,9bd75e09e53d48fe84786458f96789e7,3,2026-06-18 18:43:56.824000+00:00
6009,8630e260817e4b5f969e6e647d84c00c,2,2026-06-18 18:43:56.614000+00:00
6010,a5e5fe9e885949c4b1f63aad8de57eb7,1,2026-06-18 18:43:54.693000+00:00
6011,df2876557c694c34ab9a0f6f3934194c,0,2026-06-18 18:43:53.343000+00:00
